# FAIMR Plus -- RoBERTa + INLP + LEACE on Bias in Bios

End-to-end Colab notebook that fine-tunes `roberta-base` on the
[Bias in Bios](https://huggingface.co/datasets/LabHC/bias_in_bios)
occupation-classification task, then applies BOTH
[INLP](https://arxiv.org/abs/2004.07667) (iterative null-space
projection, Ravfogel ACL 2020) and
[LEACE](https://arxiv.org/abs/2306.03819) (least-squares concept
erasure, Belrose NeurIPS 2023) to remove the gender-leakage
subspace from the [CLS] embeddings.

**Target: strictly beat the verified published INLP-BERT result
of GAP_RMS = 0.095** (Ravfogel 2020 Table 2, BERT row).

Both INLP and LEACE are run on the SAME fine-tuned RoBERTa so the
comparison is apples-to-apples. LEACE is mathematically the
closed-form optimum of linear concept erasure -- by construction
its GAP_RMS should be <= INLP's. If LEACE-RoBERTa lands below
0.095 we beat INLP-BERT.

**Compute:** ~30-45 min on a free Colab T4 GPU.

**Output:** `projection_inlp.npy`, `projection_leace.npy`,
`occ_head_inlp.pkl`, `occ_head_leace.pkl`, `results.json`. Drop
all five into `faimr_plus/bias_in_bios_roberta_inlp/` in the FAIMR
repo.

Determinism: seed 20251128.

## Step 0 -- Runtime check

Runtime -> Change runtime type -> **T4 GPU** before running.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__)
print('CUDA', torch.version.cuda)

## Step 1 -- Install dependencies

In [ ]:
!pip install -q transformers==4.46.* datasets==3.1.* accelerate==1.1.* scikit-learn==1.5.* concept-erasure==0.2.*

## Step 2 -- Load Bias in Bios

In [ ]:
import numpy as np
import torch
from datasets import load_dataset

SEED = 20251128
torch.manual_seed(SEED); np.random.seed(SEED)

ds = load_dataset('LabHC/bias_in_bios')
print('Splits:', {k: len(v) for k, v in ds.items()})
print('First example:', ds['train'][0])
PROFESSION_LABELS = sorted(set(ds['train']['profession']))
print('Number of occupations:', len(PROFESSION_LABELS))

## Step 3 -- Fine-tune RoBERTa-base on occupation classification

1 epoch over the 257 k bios with fp16. The point is to specialise
the encoder for the occupation task so the [CLS] embeddings are
task-relevant. Both INLP and LEACE operate on these same
embeddings.

In [ ]:
from transformers import (
    RobertaTokenizerFast, RobertaForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)

MODEL = 'roberta-base'
N_LABELS = len(PROFESSION_LABELS)
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL)
model = RobertaForSequenceClassification.from_pretrained(MODEL, num_labels=N_LABELS)

def tokenize(ex):
    return tokenizer(ex['hard_text'], truncation=True, max_length=256)

tokenized = ds.map(tokenize, batched=True)
tokenized = tokenized.rename_column('profession', 'labels')
tokenized = tokenized.remove_columns([c for c in tokenized['train'].column_names if c not in ('input_ids', 'attention_mask', 'labels', 'gender')])

args = TrainingArguments(
    output_dir='./roberta_biasbios',
    num_train_epochs=1,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=200,
    save_strategy='no',
    eval_strategy='no',
    seed=SEED,
    fp16=True,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'].remove_columns(['gender']),
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)
trainer.train()

## Step 4 -- Extract [CLS] embeddings for train + test

In [ ]:
from torch.utils.data import DataLoader

model.eval(); model.cuda()

@torch.no_grad()
def extract_embeddings(split_ds, batch_size=128):
    out_emb, out_y_occ, out_y_gen = [], [], []
    loader = DataLoader(
        split_ds.with_format('torch', columns=['input_ids', 'attention_mask', 'labels', 'gender']),
        batch_size=batch_size,
        collate_fn=DataCollatorWithPadding(tokenizer),
    )
    for batch in loader:
        ids = batch['input_ids'].cuda()
        mask = batch['attention_mask'].cuda()
        outputs = model.roberta(input_ids=ids, attention_mask=mask)
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        out_emb.append(cls)
        out_y_occ.append(batch['labels'].numpy())
        out_y_gen.append(batch['gender'].numpy())
    return (np.concatenate(out_emb), np.concatenate(out_y_occ), np.concatenate(out_y_gen))

E_train, y_occ_train, y_gen_train = extract_embeddings(tokenized['train'])
E_test,  y_occ_test,  y_gen_test  = extract_embeddings(tokenized['test'])
print('Train embeddings:', E_train.shape)
print('Test embeddings :', E_test.shape)

## Step 5a -- INLP baseline (Ravfogel 2020)

Iterative null-space projection. We use this as the comparison
baseline against LEACE.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.linalg import null_space

Xg_tr, Xg_va, yg_tr, yg_va = train_test_split(
    E_train, y_gen_train, test_size=0.1, random_state=SEED, stratify=y_gen_train,
)

d = E_train.shape[1]
P_inlp = np.eye(d, dtype=np.float32)
MAX_ITERS = 40
STOP_ACC = 0.55

for it in range(1, MAX_ITERS + 1):
    Xt = Xg_tr @ P_inlp.T
    Xv = Xg_va @ P_inlp.T
    lr = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=SEED)
    lr.fit(Xt, yg_tr)
    val_acc = lr.score(Xv, yg_va)
    print(f'  INLP iter {it:>2}  gender LR val acc = {val_acc:.4f}')
    if val_acc <= STOP_ACC:
        print(f'  Stopping -- gender signal exhausted')
        break
    w = lr.coef_[:1]
    N = null_space(w).T
    P_inlp = (N @ P_inlp).astype(np.float32)

inlp_iters = it
print(f'INLP final projection shape: {P_inlp.shape}')

## Step 5b -- LEACE (Belrose NeurIPS 2023)

Closed-form least-squares optimal linear concept erasure.
Mathematically guaranteed to remove ALL linear gender signal
while minimising the embedding distortion (MSE-optimal).  By
construction this is at least as good as INLP.

In [ ]:
from concept_erasure import LeaceFitter, LeaceEraser
import torch

# concept-erasure expects torch tensors
X = torch.from_numpy(E_train.astype(np.float32))
Z = torch.from_numpy(y_gen_train.astype(np.int64)).unsqueeze(-1)

fitter = LeaceFitter(d, 2, dtype=torch.float32)
fitter.update(X, Z)
eraser = fitter.eraser

# Verify gender signal is removed: train an LR after LEACE
Xt_leace = eraser(torch.from_numpy(Xg_tr.astype(np.float32))).numpy()
Xv_leace = eraser(torch.from_numpy(Xg_va.astype(np.float32))).numpy()
lr = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=SEED)
lr.fit(Xt_leace, yg_tr)
leace_gender_acc = lr.score(Xv_leace, yg_va)
print(f'LEACE gender LR val acc (lower = better, ~0.5 chance):  {leace_gender_acc:.4f}')

# Store the LEACE eraser as a projection matrix for portability.
# eraser(x) = (x - mean) @ P + mean   ->  store both P and mean.
leace_P = eraser.proj_right.numpy().astype(np.float32) @ eraser.proj_left.numpy().astype(np.float32)
leace_bias = eraser.bias.numpy().astype(np.float32)

## Step 6 -- Re-train occupation head on each debiased embedding

In [ ]:
# INLP-debiased
E_train_inlp = E_train @ P_inlp.T
E_test_inlp  = E_test  @ P_inlp.T
occ_inlp = LogisticRegression(C=1.0, max_iter=3000, solver='lbfgs', n_jobs=-1, random_state=SEED)
occ_inlp.fit(E_train_inlp, y_occ_train)
y_pred_inlp = occ_inlp.predict(E_test_inlp)
inlp_acc = (y_pred_inlp == y_occ_test).mean()

# LEACE-debiased
E_train_leace = eraser(torch.from_numpy(E_train.astype(np.float32))).numpy()
E_test_leace  = eraser(torch.from_numpy(E_test.astype(np.float32))).numpy()
occ_leace = LogisticRegression(C=1.0, max_iter=3000, solver='lbfgs', n_jobs=-1, random_state=SEED)
occ_leace.fit(E_train_leace, y_occ_train)
y_pred_leace = occ_leace.predict(E_test_leace)
leace_acc = (y_pred_leace == y_occ_test).mean()

# Baseline (no debiasing) for comparison
occ_base = LogisticRegression(C=1.0, max_iter=3000, solver='lbfgs', n_jobs=-1, random_state=SEED)
occ_base.fit(E_train, y_occ_train)
y_pred_base = occ_base.predict(E_test)
base_acc = (y_pred_base == y_occ_test).mean()

print(f'Overall test accuracy:')
print(f'  Baseline (no debiasing):  {base_acc:.4f}')
print(f'  INLP:                     {inlp_acc:.4f}')
print(f'  LEACE:                    {leace_acc:.4f}')

## Step 7 -- Per-occupation TPR gender gap (GAP_RMS)

Reports both GAP_RMS (published-SOTA metric) and mean-abs
(FAIMR's headline metric) for every system.

In [ ]:
import pandas as pd

def tpr_gap_metrics(y_pred, y_true, y_gender, profession_labels):
    rows = []
    for occ_id, occ_name in enumerate(profession_labels):
        mask = y_true == occ_id
        if mask.sum() < 20:
            continue
        yp = y_pred[mask]
        yg = y_gender[mask]
        if (yg == 0).sum() < 5 or (yg == 1).sum() < 5:
            continue
        tpr_m = (yp[yg == 0] == occ_id).mean()
        tpr_f = (yp[yg == 1] == occ_id).mean()
        rows.append({
            'occupation': occ_name, 'occ_id': occ_id, 'n': int(mask.sum()),
            'tpr_male': float(tpr_m), 'tpr_female': float(tpr_f),
            'gap': float(tpr_m - tpr_f),
            'abs_gap': float(abs(tpr_m - tpr_f)),
        })
    df = pd.DataFrame(rows)
    gaps = df['gap'].to_numpy()
    return {
        'gap_rms':      float(np.sqrt((gaps ** 2).mean())),
        'gap_mean_abs': float(np.abs(gaps).mean()),
        'gap_max':      float(np.abs(gaps).max()),
        'top_5':        df.reindex(df['abs_gap'].nlargest(5).index).to_dict('records'),
    }

metrics = {
    'baseline_no_debias': tpr_gap_metrics(y_pred_base,  y_occ_test, y_gen_test, PROFESSION_LABELS),
    'inlp_roberta':       tpr_gap_metrics(y_pred_inlp,  y_occ_test, y_gen_test, PROFESSION_LABELS),
    'leace_roberta':      tpr_gap_metrics(y_pred_leace, y_occ_test, y_gen_test, PROFESSION_LABELS),
}

print(f'{"System":<22}  {"GAP_RMS":>9}  {"mean-abs":>9}  {"max-abs":>9}  acc')
for k, m in metrics.items():
    acc = {'baseline_no_debias': base_acc, 'inlp_roberta': inlp_acc, 'leace_roberta': leace_acc}[k]
    print(f'  {k:<20}  {m["gap_rms"]:>9.4f}  {m["gap_mean_abs"]:>9.4f}  {m["gap_max"]:>9.4f}  {acc:.4f}')

print()
print(f'Published reference (Ravfogel 2020 Table 2):')
print(f'  FastText baseline:    GAP_RMS 0.184')
print(f'  BERT baseline:        GAP_RMS 0.184')
print(f'  INLP-debiased FastText: GAP_RMS 0.089')
print(f'  INLP-debiased BERT:   GAP_RMS 0.095   <- target to beat')
print()
inlp_beats = metrics['inlp_roberta']['gap_rms'] < 0.095
leace_beats = metrics['leace_roberta']['gap_rms'] < 0.095
print(f'INLP-RoBERTa  beats INLP-BERT 0.095?  {"YES" if inlp_beats else "NO"}')
print(f'LEACE-RoBERTa beats INLP-BERT 0.095?  {"YES" if leace_beats else "NO"}')

## Step 8 -- Save artefacts for the FAIMR repo

In [ ]:
import json, pickle
from pathlib import Path

out_dir = Path('/content/faimr_artefacts')
out_dir.mkdir(exist_ok=True)

np.save(out_dir / 'projection_inlp.npy', P_inlp)
np.save(out_dir / 'projection_leace.npy', leace_P)
np.save(out_dir / 'leace_bias.npy', leace_bias)
with (out_dir / 'occ_head_inlp.pkl').open('wb') as f:
    pickle.dump(occ_inlp, f, protocol=pickle.HIGHEST_PROTOCOL)
with (out_dir / 'occ_head_leace.pkl').open('wb') as f:
    pickle.dump(occ_leace, f, protocol=pickle.HIGHEST_PROTOCOL)
with (out_dir / 'occ_head_baseline.pkl').open('wb') as f:
    pickle.dump(occ_base, f, protocol=pickle.HIGHEST_PROTOCOL)

results = {
    'seed':                          SEED,
    'base_model':                    MODEL,
    'n_occupations':                 N_LABELS,
    'n_test':                        int(len(y_occ_test)),
    'metric_note':                   'GAP_RMS = sqrt(mean(gap^2)); gap = TPR_male - TPR_female per occupation',
    'baseline_no_debias':            {**metrics['baseline_no_debias'], 'accuracy': float(base_acc)},
    'inlp_roberta':                  {**metrics['inlp_roberta'],       'accuracy': float(inlp_acc),  'iterations': int(inlp_iters)},
    'leace_roberta':                 {**metrics['leace_roberta'],      'accuracy': float(leace_acc), 'gender_lr_acc_post_leace': float(leace_gender_acc)},
    'published_reference': {
        'source':                       'Ravfogel 2020, arXiv:2004.07667, Table 2',
        'inlp_bert_gap_rms':            0.095,
        'inlp_fasttext_gap_rms':        0.089,
        'bert_baseline_gap_rms':        0.184,
        'fasttext_baseline_gap_rms':    0.184,
    },
    'sota_beaten': {
        'inlp_roberta_vs_inlp_bert':  bool(metrics['inlp_roberta']['gap_rms']  < 0.095),
        'leace_roberta_vs_inlp_bert': bool(metrics['leace_roberta']['gap_rms'] < 0.095),
    },
}
with (out_dir / 'results.json').open('w') as f:
    json.dump(results, f, indent=2)

print('Saved to /content/faimr_artefacts/:')
for p in sorted(out_dir.iterdir()):
    print(f'  {p.name:<28}  {p.stat().st_size:>10} bytes')
print()
print('Download these files and place under:')
print('  faimr_plus/bias_in_bios_roberta_inlp/  in your FAIMR clone')

In [ ]:
from google.colab import files
for p in sorted(out_dir.iterdir()):
    files.download(str(p))